# Physical AI 最終課題
## AI気分理解ロボット・コレオグラファー

自然言語のメッセージをAI分類器が解釈し、ROS 2を通じてCRANE+ V2の振付を選択する。

- AI: TF-IDF character n-gram + Logistic Regression
- Labels: celebrate / calm / sympathy
- ROS 2: AI推論ノードとロボット制御ノードをTopicで接続
- Simulator: Gazebo + CRANE+ V2

このColabでは、AI分類器の評価、学習済みモデル、ROS 2パッケージ、ZIPを生成する。

In [ ]:
!pip -q install scikit-learn pandas matplotlib seaborn joblib

In [ ]:

import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import joblib

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

celebrate_texts = [
    "合格した！",
    "やったー、成功した！",
    "今日は最高の日だ",
    "誕生日を祝ってほしい",
    "頑張ったから褒めて！",
    "目標を達成できた",
    "嬉しいことがあった",
    "みんなで盛り上がろう",
    "試合に勝った！",
    "いいニュースが届いた",
    "営業目標を達成した！",
    "初めて大きな契約が取れた",
    "昇進が決まった",
    "希望していた会社に転職できた",
    "プレゼンが大成功だった",
    "プロジェクトを無事に完遂した",
    "資格試験に合格した",
    "お客様から感謝の言葉をもらった",
    "自分の企画が採用された！",
    "起業して初めて利益が出た",
    "子供が初めて歩いた！",
    "子供が自転車に乗れるようになった",
    "子供が学校で褒められた",
    "家族みんなで楽しい旅行ができた",
    "子供が自分からお手伝いしてくれた",
    "子供が志望校に合格した！",
    "家族で誕生日をお祝いした",
    "子供の運動会で成長を感じた",
    "今日は家族とたくさん笑えた",
    "育児を頑張った自分を褒めたい",
    "テストで100点を取った！",
    "かけっこで一番になった",
    "友達がたくさんできた",
    "逆上がりができるようになった",
    "先生に頑張りを褒められた",
    "定期テストの順位が上がった",
    "部活の試合で勝った！",
    "英検に合格した",
    "合唱コンクールで優勝した",
    "文化祭が大成功だった",
    "第一志望の大学に合格した！",
    "部活のレギュラーに選ばれた",
    "模試で目標の成績を取れた",
    "文化祭でクラスの企画が成功した",
    "アルバイトで初めて給料をもらった",
    "第一志望の企業から内定をもらった",
    "卒業論文を書き終えた！",
    "研究発表がうまくいった",
    "念願の留学が決まった",
    "試験で必要な単位を取れた",
    "地域のお祭りが無事に成功した",
    "ボランティア活動で感謝された",
    "長年続けてきた趣味で成果が出た",
    "チームのみんなで目標を達成した",
    "新しいことに挑戦して結果を出せた",
]

calm_texts = [
    "少し疲れたので休みたい",
    "気持ちを落ち着かせたい",
    "静かに過ごしたい",
    "リラックスしたいな",
    "深呼吸して一息つきたい",
    "穏やかな気分になりたい",
    "ゆっくり休ませて",
    "静かな時間がほしい",
    "のんびり過ごしたい",
    "少し気分転換したい",
    "会議続きで頭が疲れた",
    "仕事が終わったので一息つきたい",
    "通勤ラッシュで疲れた",
    "今日は仕事のことを忘れたい",
    "在宅勤務の合間に休憩したい",
    "接客で気を使ったので静かにしたい",
    "夜勤明けなのでゆっくりしたい",
    "仕事の緊張をほぐしたい",
    "家事が終わったので一休みしたい",
    "休日は何もせずのんびりしたい",
    "子供が寝たのでやっと一息つける",
    "育児の合間に少し休みたい",
    "家事を終えてゆっくりしたい",
    "子供と穏やかな時間を過ごしたい",
    "寝かしつけが終わったので休憩したい",
    "家族でのんびり過ごす休日にしたい",
    "少しだけ一人の時間がほしい",
    "夫婦でゆっくりお茶を飲みたい",
    "子供と静かに絵本を読みたい",
    "仕事と子育ての疲れを癒やしたい",
    "学校から帰ってきたので休みたい",
    "宿題の前にちょっと休憩",
    "静かに本を読みたい",
    "外で遊んだから一休みしたい",
    "お昼寝したいな",
    "部活で疲れたから休みたい",
    "テスト勉強の合間に一息つきたい",
    "塾が終わったのでのんびりしたい",
    "スマホを置いてぼーっとしたい",
    "明日の試験に備えてゆっくりしたい",
    "受験勉強で疲れたので休憩したい",
    "模試が終わったから一息つきたい",
    "部活と勉強の合間に休みたい",
    "放課後は静かに過ごしたい",
    "寝る前にゆったりした気分になりたい",
    "レポートを書き終えたので休憩したい",
    "講義が続いて疲れた",
    "就活の合間にリフレッシュしたい",
    "研究室で集中し続けて疲れた",
    "アルバイトの後はゆっくりしたい",
    "介護の合間に少し休みたい",
    "長時間の運転で疲れたので休憩したい",
    "旅行先でのんびり景色を眺めたい",
    "一日の終わりに静かな音楽を聴きたい",
    "忙しさから離れてゆったり過ごしたい",
]

sympathy_texts = [
    "失敗して落ち込んでいる",
    "元気が出ない",
    "悲しい気分です",
    "励ましてほしい",
    "今日は本当につらかった",
    "うまくいかなくて悔しい",
    "少し不安です",
    "自信をなくしてしまった",
    "寂しいので話を聞いてほしい",
    "誰かに味方になってほしい",
    "仕事で大きなミスをしてしまった",
    "上司に叱られて落ち込んでいる",
    "頑張っているのに評価されない",
    "転職活動がうまくいかなくてつらい",
    "職場で孤立している気がする",
    "責任が重くてプレッシャーを感じる",
    "部下との関係に悩んでいる",
    "仕事が多すぎて気持ちに余裕がない",
    "就職先がなかなか決まらない",
    "将来の生活が不安で仕方ない",
    "子育てが思いどおりにいかなくてつらい",
    "子供に強く叱りすぎて後悔している",
    "育児の悩みを誰にも相談できない",
    "仕事と子育ての両立が大変",
    "自分の時間がなくて疲れ切っている",
    "子供の将来が心配でたまらない",
    "ほかの親と比べて自信をなくした",
    "子供が学校に行きたがらず心配",
    "家族のために頑張っているのに報われない",
    "親としてこれでいいのか不安になる",
    "テストで悪い点を取っちゃった",
    "友達に仲間外れにされた",
    "先生に怒られて悲しかった",
    "頑張ったのにかけっこで負けた",
    "友達とけんかしてしまった",
    "友達との関係がうまくいかない",
    "テストの点数が悪くて落ち込んでいる",
    "部活で自分だけ上達しない",
    "クラスに居場所がない気がする",
    "受験に失敗しそうで不安",
    "受験勉強を頑張っても成績が上がらない",
    "進路をどう決めたらいいかわからない",
    "模試の結果が悪くて落ち込んでいる",
    "周りが進路を決めていて焦る",
    "将来やりたいことが見つからない",
    "就職活動がうまくいかなくて不安",
    "友達が次々と内定をもらって焦る",
    "卒業論文が進まなくて困っている",
    "大学で友達ができず寂しい",
    "周りと比べて自分に自信が持てない",
    "介護と仕事の両立がつらい",
    "地域で相談できる人がいなくて寂しい",
    "長い間努力してきたのに結果が出ない",
    "大切な人との別れがつらい",
    "何をやってもうまくいかない気がする",
]

training_data = []
training_data += [(text, "celebrate") for text in celebrate_texts]
training_data += [(text, "calm") for text in calm_texts]
training_data += [(text, "sympathy") for text in sympathy_texts]

ambiguous_test_data = [
    ("試験が終わった", ["calm", "celebrate"]),
    ("仕事が終わった", ["calm", "celebrate"]),
    ("やっと家に帰れる", ["calm", "celebrate"]),
    ("今日は休みだ", ["calm", "celebrate"]),
    ("長い一日だった", ["calm", "sympathy"]),
    ("結果が出た", ["celebrate", "sympathy"]),
    ("試合が終わった", ["celebrate", "sympathy"]),
    ("先生に呼ばれた", ["celebrate", "sympathy"]),
    ("親に話したいことがある", ["celebrate", "sympathy"]),
    ("明日が大事な日だ", ["celebrate", "sympathy"]),
    ("一人になりたい", ["calm", "sympathy"]),
    ("何もしたくない", ["calm", "sympathy"]),
    ("今日は静かにしていたい", ["calm", "sympathy"]),
    ("少し休ませてほしい", ["calm", "sympathy"]),
    ("誰とも話したくない", ["calm", "sympathy"]),
    ("まあ、いろいろあった", ["celebrate", "calm", "sympathy"]),
    ("これでよかったのかな", ["calm", "sympathy"]),
    ("やっと終わった", ["calm", "celebrate", "sympathy"]),
    ("明日から新しい生活が始まる", ["celebrate", "sympathy"]),
    ("頑張ったよ", ["celebrate", "sympathy"]),
]

print("Training data size:", len(training_data))
print("Celebrate:", len(celebrate_texts))
print("Calm:", len(calm_texts))
print("Sympathy:", len(sympathy_texts))
print("Ambiguous test data:", len(ambiguous_test_data))


Training data size: 165
Celebrate: 55
Calm: 55
Sympathy: 55
Ambiguous test data: 20


In [ ]:

X_train = [text for text, label in training_data]
y_train = [label for text, label in training_data]

model = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            analyzer="char",
            ngram_range=(2, 4),
            sublinear_tf=True,
        ),
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=3000,
            random_state=SEED,
        ),
    ),
])

model.fit(X_train, y_train)

print("AI classifier training completed.")
print("Classes:", list(model.named_steps["classifier"].classes_))


AI classifier training completed.
Classes: [np.str_('calm'), np.str_('celebrate'), np.str_('sympathy')]


In [ ]:
def predict_with_probability(text):
    predicted_label = model.predict([text])[0]

    probabilities = model.predict_proba([text])[0]

    classes = model.named_steps["classifier"].classes_

    result_table = pd.DataFrame({
        "label": classes,
        "probability": probabilities,
    }).sort_values(
        "probability",
        ascending=False,
    )

    return predicted_label, result_table


# 英語の内部ラベルを、日本語表示名へ変換する
label_jp = {
    "celebrate": "祝福・喜び",
    "calm": "休息・安らぎ",
    "sympathy": "共感・励まし",
}

rows = []

for text, acceptable_labels in ambiguous_test_data:
    prediction, probabilities = predict_with_probability(text)

    rows.append({
        "入力文": text,
        "AI予測": prediction,
        "AI予測（日本語）": label_jp[prediction],
        "許容ラベル": ", ".join(acceptable_labels),
        "許容ラベル（日本語）": ", ".join(
            label_jp[label] for label in acceptable_labels
        ),
        "判定": prediction in acceptable_labels,
        "確信度": probabilities.iloc[0]["probability"],
    })

ambiguous_result = pd.DataFrame(rows)

display(ambiguous_result)

acceptable_rate = ambiguous_result["判定"].mean()

print(f"曖昧表現に対する許容率: {acceptable_rate:.3f}")
print(f"許容された件数: {ambiguous_result['判定'].sum()} / {len(ambiguous_result)}")


,入力文,AI予測,AI予測（日本語）,許容ラベル,許容ラベル（日本語）,判定,確信度
0,試験が終わった,calm,休息・安らぎ,"calm, celebrate","休息・安らぎ, 祝福・喜び",True,0.477450
1,仕事が終わった,calm,休息・安らぎ,"calm, celebrate","休息・安らぎ, 祝福・喜び",True,0.484865
2,やっと家に帰れる,celebrate,祝福・喜び,"calm, celebrate","休息・安らぎ, 祝福・喜び",True,0.346780
3,今日は休みだ,celebrate,祝福・喜び,"calm, celebrate","休息・安らぎ, 祝福・喜び",True,0.362129
4,長い一日だった,celebrate,祝福・喜び,"calm, sympathy","休息・安らぎ, 共感・励まし",False,0.450631
5,結果が出た,celebrate,祝福・喜び,"celebrate, sympathy","祝福・喜び, 共感・励まし",True,0.397375
6,試合が終わった,calm,休息・安らぎ,"celebrate, sympathy","祝福・喜び, 共感・励まし",False,0.466235
7,先生に呼ばれた,celebrate,祝福・喜び,"celebrate, sympathy","祝福・喜び, 共感・励まし",True,0.414795
8,親に話したいことがある,celebrate,祝福・喜び,"celebrate, sympathy","祝福・喜び, 共感・励まし",True,0.372501
9,明日が大事な日だ,celebrate,祝福・喜び,"celebrate, sympathy","祝福・喜び, 共感・励まし",True,0.395319


曖昧表現に対する許容率: 0.900
許容された件数: 18 / 20


In [ ]:

from pathlib import Path
import shutil
import os

PROJECT_NAME = "final_project_choreographer"
ROOT = Path(PROJECT_NAME)
PACKAGE_DIR = ROOT / PROJECT_NAME

if ROOT.exists():
    shutil.rmtree(ROOT)

PACKAGE_DIR.mkdir(parents=True, exist_ok=True)
(ROOT / "resource").mkdir(parents=True, exist_ok=True)

joblib.dump(model, ROOT / "model.joblib")

mood_ai_lines = [
    "#!/usr/bin/env python3",
    "from pathlib import Path",
    "import joblib",
    "import rclpy",
    "from rclpy.node import Node",
    "from std_msgs.msg import String",
    "from ament_index_python.packages import get_package_share_directory",
    "",
    "class MoodAINode(Node):",
    "    def __init__(self):",
    "        super().__init__('mood_ai_node')",
    "        self.declare_parameter('prompt', '今日も頑張ったから褒めて！')",
    "        self.publisher = self.create_publisher(String, '/choreography_label', 10)",
    "        package_dir = get_package_share_directory('final_project_choreographer')",
    "        model_path = Path(package_dir) / 'model.joblib'",
    "        self.model = joblib.load(model_path)",
    "        prompt = self.get_parameter('prompt').value",
    "        label = self.model.predict([prompt])[0]",
    "        confidence = float(max(self.model.predict_proba([prompt])[0]))",
    "        self.get_logger().info(f'Input: {prompt}')",
    "        self.get_logger().info(f'Prediction: {label}, confidence={confidence:.3f}')",
    "        self.message = String()",
    "        self.message.data = label",
    "        self.count = 0",
    "        self.timer = self.create_timer(0.5, self.publish_label)",
    "",
    "    def publish_label(self):",
    "        self.publisher.publish(self.message)",
    "        self.count += 1",
    "        self.get_logger().info(f'Published: {self.message.data} ({self.count}/3)')",
    "        if self.count >= 3:",
    "            self.timer.cancel()",
    "",
    "def main(args=None):",
    "    rclpy.init(args=args)",
    "    node = MoodAINode()",
    "    try:",
    "        rclpy.spin(node)",
    "    except KeyboardInterrupt:",
    "        pass",
    "    finally:",
    "        node.destroy_node()",
    "        rclpy.shutdown()",
    "",
    "if __name__ == '__main__':",
    "    main()",
]

motion_executor_lines = [
    "#!/usr/bin/env python3",
    "import time",
    "import rclpy",
    "from rclpy.node import Node",
    "from std_msgs.msg import String",
    "from trajectory_msgs.msg import JointTrajectory, JointTrajectoryPoint",
    "from builtin_interfaces.msg import Duration",
    "",
    "class MotionExecutor(Node):",
    "    def __init__(self):",
    "        super().__init__('motion_executor')",
    "        self.publisher = self.create_publisher(",
    "            JointTrajectory,",
    "            '/crane_plus_arm_controller/joint_trajectory',",
    "            10,",
    "        )",
    "        self.subscription = self.create_subscription(",
    "            String,",
    "            '/choreography_label',",
    "            self.callback,",
    "            10,",
    "        )",
    "        self.executed = False",
    "        self.get_logger().info('Waiting for /choreography_label')",
    "",
    "    def callback(self, msg):",
    "        if self.executed:",
    "            return",
    "        self.executed = True",
    "        label = msg.data.strip().lower()",
    "        self.get_logger().info(f'Received label: {label}')",
    "        if label == 'celebrate':",
    "            self.celebrate()",
    "        elif label == 'calm':",
    "            self.calm()",
    "        elif label == 'sympathy':",
    "            self.sympathy()",
    "        else:",
    "            self.home()",
    "",
    "    def move_to(self, positions, duration=2.0):",
    "        msg = JointTrajectory()",
    "        msg.joint_names = [",
    "            'crane_plus_joint1',",
    "            'crane_plus_joint2',",
    "            'crane_plus_joint3',",
    "            'crane_plus_joint4',",
    "        ]",
    "        point = JointTrajectoryPoint()",
    "        point.positions = positions",
    "        point.velocities = [0.0, 0.0, 0.0, 0.0]",
    "        point.accelerations = [0.0, 0.0, 0.0, 0.0]",
    "        point.time_from_start = Duration(",
    "            sec=int(duration),",
    "            nanosec=int((duration % 1.0) * 1000000000),",
    "        )",
    "        msg.points.append(point)",
    "        self.publisher.publish(msg)",
    "        time.sleep(duration + 0.3)",
    "",
    "    def home(self):",
    "        self.move_to([0.0, 0.0, 0.0, 0.0], 2.0)",
    "",
    "    def celebrate(self):",
    "        self.home()",
    "        self.move_to([0.55, 0.25, -0.25, 0.15], 1.4)",
    "        self.move_to([-0.55, 0.25, -0.25, 0.15], 1.4)",
    "        self.move_to([0.55, 0.25, -0.25, 0.15], 1.4)",
    "        self.move_to([-0.55, 0.25, -0.25, 0.15], 1.4)",
    "        self.home()",
    "",
    "    def calm(self):",
    "        self.home()",
    "        self.move_to([0.20, 0.15, -0.15, 0.10], 2.2)",
    "        self.move_to([-0.20, 0.15, -0.15, 0.10], 2.2)",
    "        self.move_to([0.15, 0.05, -0.05, 0.05], 2.2)",
    "        self.move_to([-0.15, 0.05, -0.05, 0.05], 2.2)",
    "        self.home()",
    "",
    "    def sympathy(self):",
    "        self.home()",
    "        self.move_to([0.0, 0.50, -0.40, 0.25], 2.5)",
    "        self.move_to([0.0, 0.35, -0.30, 0.18], 1.8)",
    "        self.move_to([0.22, 0.25, -0.25, 0.15], 1.6)",
    "        self.move_to([-0.22, 0.25, -0.25, 0.15], 1.6)",
    "        self.home()",
    "",
    "def main(args=None):",
    "    rclpy.init(args=args)",
    "    node = MotionExecutor()",
    "    try:",
    "        rclpy.spin(node)",
    "    except KeyboardInterrupt:",
    "        pass",
    "    finally:",
    "        node.destroy_node()",
    "        rclpy.shutdown()",
    "",
    "if __name__ == '__main__':",
    "    main()",
]

setup_py_lines = [
    "from setuptools import find_packages, setup",
    "",
    "package_name = 'final_project_choreographer'",
    "",
    "setup(",
    "    name=package_name,",
    "    version='0.0.0',",
    "    packages=find_packages(exclude=['test']),",
    "    data_files=[",
    "        ('share/ament_index/resource_index/packages', ['resource/' + package_name]),",
    "        ('share/' + package_name, ['package.xml', 'model.joblib']),",
    "    ],",
    "    install_requires=['setuptools'],",
    "    zip_safe=True,",
    "    maintainer='YOUR_NAME',",
    "    maintainer_email='YOUR_EMAIL@example.com',",
    "    description='AI-driven robot choreographer for Physical AI final project',",
    "    license='MIT',",
    "    entry_points={",
    "        'console_scripts': [",
    "            'mood_ai_node = final_project_choreographer.mood_ai_node:main',",
    "            'motion_executor = final_project_choreographer.motion_executor:main',",
    "        ],",
    "    },",
    ")",
]

setup_cfg_lines = [
    "[develop]",
    "script_dir=$base/lib/final_project_choreographer",
    "",
    "[install]",
    "install_scripts=$base/lib/final_project_choreographer",
]

package_xml_lines = [
    '<?xml version="1.0"?>',
    '<package format="3">',
    "  <name>final_project_choreographer</name>",
    "  <version>0.0.0</version>",
    "  <description>AI-driven CRANE+ robot choreographer for Physical AI final project.</description>",
    '  <maintainer email="YOUR_EMAIL@example.com">YOUR_NAME</maintainer>',
    "  <license>MIT</license>",
    "  <depend>rclpy</depend>",
    "  <depend>std_msgs</depend>",
    "  <depend>trajectory_msgs</depend>",
    "  <depend>builtin_interfaces</depend>",
    "  <depend>ament_index_python</depend>",
    "  <export>",
    "    <build_type>ament_python</build_type>",
    "  </export>",
    "</package>",
]

readme_lines = [
    "# AI Mood-Aware Robot Choreographer",
    "",
    "Physical AI応用1講座・最終課題用ROS 2パッケージ。",
    "",
    "## Labels",
    "- celebrate: 祝福・喜び",
    "- calm: 休息・安らぎ",
    "- sympathy: 共感・励まし",
    "",
    "## Required",
    "- ROS 2 Humble",
    "- Gazebo",
    "- CRANE+ V2 Gazebo package",
    "- Python scikit-learn",
    "",
    "## Build",
    "cd ~/pai_ws",
    "python3 -m pip install --user scikit-learn joblib",
    "colcon build --symlink-install --packages-select final_project_choreographer",
]

(PACKAGE_DIR / "mood_ai_node.py").write_text(
    "\n".join(mood_ai_lines) + "\n",
    encoding="utf-8",
)

(PACKAGE_DIR / "motion_executor.py").write_text(
    "\n".join(motion_executor_lines) + "\n",
    encoding="utf-8",
)

(PACKAGE_DIR / "__init__.py").write_text("", encoding="utf-8")
(ROOT / "resource" / PROJECT_NAME).write_text("", encoding="utf-8")

(ROOT / "setup.py").write_text(
    "\n".join(setup_py_lines) + "\n",
    encoding="utf-8",
)

(ROOT / "setup.cfg").write_text(
    "\n".join(setup_cfg_lines) + "\n",
    encoding="utf-8",
)

(ROOT / "package.xml").write_text(
    "\n".join(package_xml_lines) + "\n",
    encoding="utf-8",
)

(ROOT / "README.md").write_text(
    "\n".join(readme_lines) + "\n",
    encoding="utf-8",
)

(ROOT / "requirements.txt").write_text(
    "scikit-learn>=1.0\njoblib>=1.0\n",
    encoding="utf-8",
)

os.chmod(PACKAGE_DIR / "mood_ai_node.py", 0o755)
os.chmod(PACKAGE_DIR / "motion_executor.py", 0o755)

print("ROS 2 package generated successfully.")

for path in sorted(ROOT.rglob("*")):
    print(path)


ROS 2 package generated successfully.
final_project_choreographer/README.md
final_project_choreographer/final_project_choreographer
final_project_choreographer/final_project_choreographer/__init__.py
final_project_choreographer/final_project_choreographer/mood_ai_node.py
final_project_choreographer/final_project_choreographer/motion_executor.py
final_project_choreographer/model.joblib
final_project_choreographer/package.xml
final_project_choreographer/requirements.txt
final_project_choreographer/resource
final_project_choreographer/resource/final_project_choreographer
final_project_choreographer/setup.cfg
final_project_choreographer/setup.py


In [ ]:
zip_path = shutil.make_archive(
    'final_project_choreographer',
    'zip',
    root_dir='.',
    base_dir='final_project_choreographer',
)

print('Created:', zip_path)

from google.colab import files
files.download(zip_path)


Created: /content/final_project_choreographer.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Docker環境での実行

Colabからダウンロードした `final_project_choreographer.zip` を、ROS 2 Docker環境へ置く。

```bash
cd ~/pai_ws/src
unzip final_project_choreographer.zip

cd ~/pai_ws
python3 -m pip install --user scikit-learn joblib
source /opt/ros/humble/setup.bash
colcon build --symlink-install --packages-select final_project_choreographer
source install/setup.bash
```

Gazebo起動：

```bash
ros2 launch crane_plus_gazebo crane_plus_with_table.launch.py
```

別ターミナルでロボット振付ノードを起動：

```bash
source ~/pai_ws/install/setup.bash
ros2 run final_project_choreographer motion_executor
```

さらに別ターミナルでAI推論を実行：

```bash
source ~/pai_ws/install/setup.bash
ros2 run final_project_choreographer mood_ai_node --ros-args -p prompt:='営業目標を達成した！'
```
